# 🚀 ONE-CELL RETRAIN
Copy-paste cell di bawah. Tunggu 30-50 menit.

In [ ]:
import kagglehub, os, shutil, yaml, xml.etree.ElementTree as ET, time
from pathlib import Path
from ultralytics import YOLO

# ========== 1. DOWNLOAD ==========
print("📥 Step 1: Download dataset...")
src = Path(kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan"))
print(f"   Source: {src}")

# Show what we got
for split in ['train','valid','test']:
    d = src/split
    if d.exists():
        files = list(d.iterdir())
        print(f"   {split}/: {len(files)} items")

# ========== 2. PREPARE ==========
print("
🔄 Step 2: Prepare dataset...")
if not Path('jalancerdas-ai').exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git

CID = {'retak_memanjang':0,'pengelupasan_lapisan_permukaan':1,'lubang':2,'retak_kulit_buaya':3,'retak_blok':4,'retak_pinggir':5}
ds = Path('jalancerdas-ai/ai-model/dataset')
if ds.exists(): shutil.rmtree(ds)

total_i, total_l = 0, 0
for ss, dsplit in [('train','train'),('valid','val'),('test','test')]:
    sd = src/ss
    if not sd.exists(): print(f"   ⚠️ {ss}/ not found"); continue
    
    di = ds/dsplit/'images'; dl = ds/dsplit/'labels'
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    
    # Find images - check multiple levels
    imgs = []
    for ext in ['*.jpg','*.jpeg','*.png','*.bmp']:
        imgs.extend(sd.rglob(ext))
    
    xmls = {x.stem: x for x in sd.rglob('*.xml')}
    
    ni, nl = 0, 0
    for img in imgs:
        shutil.copy2(img, di/img.name)
        ni += 1
        xml = xmls.get(img.stem)
        if xml:
            try:
                t = ET.parse(xml); r = t.getroot()
                w = int(r.find('size/width').text)
                h = int(r.find('size/height').text)
                lines = []
                for o in r.findall('object'):
                    nm = o.find('name').text
                    if nm not in CID: continue
                    b = o.find('bndbox')
                    x1,y1 = float(b.find('xmin').text), float(b.find('ymin').text)
                    x2,y2 = float(b.find('xmax').text), float(b.find('ymax').text)
                    lines.append(f"{CID[nm]} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
                (dl/(img.stem+'.txt')).write_text('
'.join(lines))
                nl += len(lines)
            except: pass
    
    total_i += ni; total_l += nl
    print(f"   {dsplit}: {ni} images, {nl} labels")

print(f"   Total: {total_i} images, {total_l} labels")

# ========== 3. VERIFY ==========
print("
✅ Step 3: Verify...")
for s in ['train','val','test']:
    p = ds/s/'images'
    n = len(list(p.glob('*'))) if p.exists() else 0
    status = '✅' if n > 0 else '❌'
    print(f"   {s}/images: {n} files {status}")

if total_i == 0:
    print("
❌ ERROR: No images found! Check source dataset structure.")
    print("   Source contents:")
    for item in sorted(src.rglob('*'))[:20]:
        print(f"     {item.relative_to(src)}")
else:
    # ========== 4. FIX PATH ==========
    yp = Path('jalancerdas-ai/ai-model/configs/data.yaml')
    dp = str(Path.cwd()/'jalancerdas-ai/ai-model/dataset')
    with open(yp) as f: d = yaml.safe_load(f)
    d['path'] = dp
    with open(yp,'w') as f: yaml.dump(d, f, default_flow_style=False)
    print(f"
   data.yaml: {dp}")
    
    # ========== 5. TRAIN ==========
    print("
🚀 Step 4: Training YOLOv8s (200 epochs)...")
    print("   30-50 minutes on GPU T4")
    print("   DO NOT CLOSE THIS TAB!
")
    
    start = time.time()
    model = YOLO('yolov8s.pt')
    results = model.train(
        data=str(yp), epochs=200, batch=16, imgsz=640, device=0,
        patience=50, optimizer='SGD', lr0=0.01, lrf=0.01,
        mosaic=1.0, mixup=0.15, degrees=10.0, translate=0.1,
        scale=0.5, fliplr=0.5, copy_paste=0.1, close_mosaic=15,
        project='jalancerdas-ai/ai-model/runs', name='train_v2', exist_ok=True
    )
    
    elapsed = time.time() - start
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)
    
    best = 'jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt'
    print(f"
{'='*50}")
    print(f"  ✅ TRAINING DONE in {mins}m {secs}s")
    print(f"  Best: {best}")
    print(f"{'='*50}")
    
    # ========== 6. EVALUATE ==========
    print("
📊 Step 5: Evaluate...")
    model = YOLO(best)
    r = model.val(data=str(yp), device=0)
    print(f"  mAP50={r.box.map50:.4f} | Prec={r.box.mp:.4f} | Recall={r.box.mr:.4f}")
    for i,ap in enumerate(r.box.ap50):
        print(f"    {r.names[i]}: {ap:.4f}")
    
    # ========== 7. EXPORT ==========
    print("
📦 Step 6: Export TFLite...")
    ep = model.export(format='tflite', imgsz=640, half=True, simplify=True)
    sz = os.path.getsize(ep)/1024/1024
    print(f"  ✅ {ep} ({sz:.1f} MB)")
    
    # ========== 8. DOWNLOAD ==========
    print("
📥 Step 7: Download files...")
    from google.colab import files
    files.download(ep)
    files.download(best)
    print("
🎉 ALL DONE! Copy to mobile/assets/models/pothole_yolo.tflite")


## 📋 After Download
